In [ ]:
import pypsa
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from pypsa.plot import add_legend_lines, add_legend_patches, add_legend_semicircles
import yaml
from pathlib import Path
import pandas as pd
import yaml
from clusters.add_renewables_cluster import *

**Set Up**

In [172]:
fn = 'resources/DK_test/networks/base_s_2__12h_2050.nc'


In [173]:
n= pypsa.Network(fn)

config = yaml.safe_load(Path("config/config.denmark.yaml").read_text())


INFO:pypsa.network.io:New version 1.0.7 available! (Current: 0.35.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, links, loads, stores


In [203]:
p = Path(fn)  
try:
    if p.exists():
        p.unlink()
        print(f"Deleted {p}")
    else:
        print(f"File not found: {p}")
except Exception as e:
    print(f"Failed to delete {p}: {e}")

Deleted resources/DK_test/networks/base_s_2__12h_2050.nc


**Options**

In [175]:
ongrid=False
cluster_cost_reduction=0
cluster_size=1000   
renewables={"solar",'solar-hsat','onwind'}

In [176]:
nodes_with_clusters = n.buses.loc[
    n.buses.index.str[:2].isin(config['countries']) &
    (n.buses['carrier'] == 'AC')
].index.tolist()




**Buses and Generators of the Cluster Addition**

In [ ]:
n= assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables, nodes_with_clusters)

                   p_max_pu p_nom_max
Generator                            
DK0 0 0 solar-hsat      NaN       NaN
                    p_max_pu      p_nom_max
Generator                                  
DK0 0 0 solar-hsat  0.132802  105728.626454
Remaining top solar-hsat capacity outside the cluster: 104728.62645358953 MW
Capacity of the last solar-hsat generator adjusted to fit cluster size: 1000.0 MW
                      bus control type  p_nom  p_nom_mod  p_nom_extendable  \
Generator                                                                    
DK0 0 0 solar-hsat  DK0 0      PQ         0.0        0.0              True   

                    p_nom_min  p_nom_max  p_min_pu  p_max_pu  ...  \
Generator                                                     ...   
DK0 0 0 solar-hsat        0.0     1000.0       0.0       1.0  ...   

                    up_time_before  down_time_before  ramp_limit_up  \
Generator                                                             
DK0 0 0 so

**Links of the Cluster Addition**

In [ ]:
n = add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid)
        

In [183]:
n.links.loc[n.links.index.str.contains("methanolisation cluster")]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 methanolisation cluster,DK0 0 H2 cluster,DK0 0 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK1 0 methanolisation cluster,DK1 0 H2 cluster,DK1 0 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


**Storages of the Cluster Addition**

In [ ]:
n = add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction)

In [185]:
n.links["reversed"] = n.links["reversed"].fillna(False).astype(bool)


**Printing to Check**

In [186]:
n.links.loc[n.links["bus1"].str.contains('methanol')]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 solid biomass biomass-to-methanol,DK0 0 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 solid biomass biomass-to-methanol,DK1 0 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
EU industry methanol,EU methanol,EU industry methanol,,industry methanol,1.0000,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK0 0 methanolisation,DK0 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 methanolisation,DK1 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
EU shipping methanol,EU methanol,EU shipping methanol,,shipping methanol,1.0000,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK0 0 methanolisation cluster,DK0 0 H2 cluster,DK0 0 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK0 0 methanol cluster,DK0 0 methanol cluster,EU methanol,,methanol,1.0000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK1 0 methanolisation cluster,DK1 0 H2 cluster,DK1 0 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [187]:
n.links.loc[n.links.index.str.contains("Electrolysis")]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 H2 Electrolysis,DK0 0,DK0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 H2 Electrolysis,DK1 0,DK1 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK0 0 H2 Electrolysis cluster,DK0 0 cluster,DK0 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK1 0 H2 Electrolysis cluster,DK1 0 cluster,DK1 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [188]:
n.links.loc[n.links["carrier"].str.contains('H2 Electrolysis')]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 H2 Electrolysis,DK0 0,DK0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 H2 Electrolysis,DK1 0,DK1 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK0 0 H2 Electrolysis cluster,DK0 0 cluster,DK0 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK1 0 H2 Electrolysis cluster,DK1 0 cluster,DK1 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [189]:
n.buses.loc[n.buses.index.str.contains("cluster")]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
Bus,,,,,,,,,,,,,,,,
DK0 0 cluster,380.0,,9.648393,55.893047,AC,MWh_el,DK0 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0
DK0 0 H2 cluster,1.0,,9.648393,55.893047,H2,MWh_LHV,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK0 0 methanol cluster,1.0,,9.648393,55.893047,methanol,MWh_th,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK0 0 battery cluster,1.0,,9.648393,55.893047,battery,MWh_el,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK1 0 cluster,380.0,,12.303316,55.515974,AC,MWh_el,DK1 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0
DK1 0 H2 cluster,1.0,,12.303316,55.515974,H2,MWh_LHV,DK1 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK1 0 methanol cluster,1.0,,12.303316,55.515974,methanol,MWh_th,DK1 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK1 0 battery cluster,1.0,,12.303316,55.515974,battery,MWh_el,DK1 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN


In [190]:
n.stores.loc[n.stores.index.str.contains("cluster")]



,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_min_pu,e_max_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
Store,,,,,,,,,,,,,,,,,,,,,
DK0 0 H2 Store cluster,DK0 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,89.430064,0.0,True,0,inf,0.0,NaN
DK0 0 battery cluster,DK0 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN
DK1 0 H2 Store cluster,DK1 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,89.430064,0.0,True,0,inf,0.0,NaN
DK1 0 battery cluster,DK1 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,6427.168612,0.0,True,0,inf,0.0,NaN


In [191]:
n.links.loc[n.links["carrier"]=='DC']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
relation/5487095-400-DC,DK0 0,DK1 0,,DC,0.976096,True,0,inf,600.0,0.0,...,1.0,0.0,relation/5487095,LINESTRING (10.505724427906852 55.365970143543...,1.0,0.560249,NaN,,False,171.53249
relation/5487095-400-DC-reversed,DK1 0,DK0 0,,DC,0.976096,True,0,inf,600.0,0.0,...,1.0,0.0,relation/5487095,LINESTRING (10.505724427906852 55.365970143543...,1.0,0.560249,NaN,,True,171.53249


In [192]:
n.stores

,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_min_pu,e_max_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
Store,,,,,,,,,,,,,,,,,,,,,
co2 atmosphere,co2 atmosphere,,co2,0.000000,0.0,True,0.0,inf,-1.0,1.0,...,0.0,0.0,0.0,0.000000,0.000000,True,0,inf,0.0,
DK0 0 co2 stored,DK0 0 co2 stored,,co2 stored,0.000000,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,247.607546,0.000000,True,0,inf,0.0,
DK1 0 co2 stored,DK1 0 co2 stored,,co2 stored,0.000000,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,247.607546,0.000000,True,0,inf,0.0,
DK0 0 co2 sequestered,DK0 0 co2 sequestered,,co2 sequestered,0.000000,0.0,True,0.0,7.374268e+08,0.0,1.0,...,-0.1,0.0,0.0,30.000000,0.000000,True,0,50.0,0.0,
DK1 0 co2 sequestered,DK1 0 co2 sequestered,,co2 sequestered,0.000000,0.0,True,0.0,8.677499e+07,0.0,1.0,...,-0.1,0.0,0.0,30.000000,0.000000,True,0,50.0,0.0,
DK0 0 gas Store,DK0 0 gas,,gas,0.000000,0.0,True,3805600.0,inf,0.0,1.0,...,0.0,0.0,0.0,17.851178,0.000000,True,0,inf,0.0,
DK1 0 gas Store,DK1 0 gas,,gas,0.000000,0.0,True,6334336.0,inf,0.0,1.0,...,0.0,0.0,0.0,17.851178,0.000000,True,0,inf,0.0,
DK0 0 H2 Store,DK0 0 H2,,H2 Store,0.000000,0.0,True,0.0,2.013398e+08,0.0,1.0,...,0.0,0.0,0.0,89.430064,0.000000,True,0,100.0,0.0,
DK1 0 H2 Store,DK1 0 H2,,H2 Store,0.000000,0.0,True,0.0,2.879317e+06,0.0,1.0,...,0.0,0.0,0.0,89.430064,0.000000,True,0,100.0,0.0,


In [193]:
n.links.loc[n.links["bus0"]=='EU methanol']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 OCGT methanol,EU methanol,DK0 0,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 OCGT methanol,EU methanol,DK1 0,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
EU industry methanol,EU methanol,EU industry methanol,,industry methanol,1.00,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
EU shipping methanol,EU methanol,EU shipping methanol,,shipping methanol,1.00,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0


In [194]:
n.carriers

,co2_emissions,color,nice_name,max_growth,max_relative_growth
Carrier,,,,,
AC,0.0,#70af1d,AC,inf,0.0
DC,0.0,#8a1caf,DC,inf,0.0
onwind,0.0,#235ebc,Onshore Wind,inf,0.0
offwind-float,0.0,#b5e2fa,Offshore Wind (Floating),inf,0.0
offwind-dc,0.0,#74c6f2,Offshore Wind (DC),inf,0.0
...,...,...,...,...,...
agriculture machinery electric,0.0,#444578,agriculture machinery electric,inf,0.0
agriculture electricity,0.0,#494778,agriculture electricity,inf,0.0
electricity,0.0,#110d63,electricity,inf,0.0


In [195]:
n.global_constraints

,type,investment_period,carrier_attribute,sense,constant,mu
GlobalConstraint,,,,,,
lv_limit,transmission_volume_expansion_limit,NaN,"AC, DC",<=,1.029195e+05,0.0
biomass limit,operational_limit,NaN,solid biomass,<=,1.219759e+07,0.0
CO2Limit,co2_atmosphere,NaN,co2_emissions,<=,0.000000e+00,0.0


In [196]:
n.loads

,bus,carrier,type,p_set,q_set,sign,active
Load,,,,,,,
DK0 0,DK0 0 low voltage,electricity,,0.000000,0.0,-1.0,True
DK1 0,DK1 0 low voltage,electricity,,0.000000,0.0,-1.0,True
DK0 0 land transport EV,DK0 0 EV battery,land transport EV,,0.000000,0.0,-1.0,True
DK1 0 land transport EV,DK1 0 EV battery,land transport EV,,0.000000,0.0,-1.0,True
DK0 0 urban central heat,DK0 0 urban central heat,urban central heat,,0.000000,0.0,-1.0,True
DK1 0 urban central heat,DK1 0 urban central heat,urban central heat,,0.000000,0.0,-1.0,True
DK0 0 solid biomass for industry,DK0 0 solid biomass for industry,solid biomass for industry,,541.095890,0.0,-1.0,True
DK1 0 solid biomass for industry,DK1 0 solid biomass for industry,solid biomass for industry,,245.433790,0.0,-1.0,True
DK0 0 gas for industry,DK0 0 gas for industry,gas for industry,,155.251142,0.0,-1.0,True


In [197]:
n.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
relation/5487095-400-DC,DK0 0,DK1 0,,DC,0.976096,True,0,inf,600.0,0.0,...,1.0,0.0,relation/5487095,LINESTRING (10.505724427906852 55.365970143543...,1.0,0.560249,NaN,,False,171.53249
DK0 0 co2 sequestered,DK0 0 co2 stored,DK0 0 co2 sequestered,,co2 sequestered,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK1 0 co2 sequestered,DK1 0 co2 stored,DK1 0 co2 sequestered,,co2 sequestered,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK0 0 OCGT,DK0 0 gas,DK0 0,,OCGT,0.430000,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK1 0 OCGT,DK1 0 gas,DK1 0,,OCGT,0.430000,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DK1 0 methanol cluster,DK1 0 methanol cluster,EU methanol,,methanol,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK0 0 battery charger cluster,DK0 0 cluster,DK0 0 battery cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK0 0 battery discharger cluster,DK0 0 battery cluster,DK0 0 cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN


In [198]:
n.links.loc[n.links.index.str.contains("cluster"),n.links.columns.str.contains("bus")]

,bus0,bus1,bus4,bus3,bus2
Link,,,,,
DK0 0 H2 Electrolysis cluster,DK0 0 cluster,DK0 0 H2 cluster,,,
DK0 0 methanolisation cluster,DK0 0 H2 cluster,DK0 0 methanol cluster,DK0 0 urban central heat,DK0 0 co2 stored,DK0 0 cluster
DK0 0 methanol cluster,DK0 0 methanol cluster,EU methanol,,,
DK1 0 H2 Electrolysis cluster,DK1 0 cluster,DK1 0 H2 cluster,,,
DK1 0 methanolisation cluster,DK1 0 H2 cluster,DK1 0 methanol cluster,DK1 0 urban central heat,DK1 0 co2 stored,DK1 0 cluster
DK1 0 methanol cluster,DK1 0 methanol cluster,EU methanol,,,
DK0 0 battery charger cluster,DK0 0 cluster,DK0 0 battery cluster,,,
DK0 0 battery discharger cluster,DK0 0 battery cluster,DK0 0 cluster,,,
DK1 0 battery charger cluster,DK1 0 cluster,DK1 0 battery cluster,,,


In [199]:
print(n.generators.loc[n.generators.index.str.contains("solar")&
    ~n.generators.index.str.contains("solar-hsat") &~n.generators.index.str.contains("solar thermal") &~n.generators.index.str.contains("solar rooftop")])

                                 bus control type  p_nom  p_nom_mod  \
Generator                                                             
DK0 0 0 solar                  DK0 0      PQ       800.2        0.0   
DK0 0 1 solar                  DK0 0      PQ       474.0        0.0   
DK0 0 2 solar                  DK0 0      PQ       283.0        0.0   
DK0 0 3 solar                  DK0 0      PQ        73.5        0.0   
DK1 0 0 solar                  DK1 0      PQ         0.0        0.0   
DK1 0 1 solar                  DK1 0      PQ       170.0        0.0   
DK1 0 2 solar                  DK1 0      PQ       354.0        0.0   
DK1 0 3 solar                  DK1 0      PQ        15.0        0.0   
DK0 0 4 solar cluster  DK0 0 cluster      PQ         0.0        0.0   
DK0 0 3 solar cluster  DK0 0 cluster      PQ         0.0        0.0   
DK1 0 4 solar cluster  DK1 0 cluster      PQ         0.0        0.0   
DK1 0 3 solar cluster  DK1 0 cluster      PQ         0.0        0.0   

     

In [200]:
print(n.generators.loc[n.generators.index.str.contains("solar cluster"), n.generators.columns.isin(['p_nom_max','carrier','location','pnom_extendable'])])


                        p_nom_max carrier location
Generator                                         
DK0 0 4 solar cluster  165.931050   solar         
DK0 0 3 solar cluster  834.068950   solar         
DK1 0 4 solar cluster  971.684185   solar         
DK1 0 3 solar cluster   28.315815   solar         


In [201]:
n.generators_t['p_max_pu'].loc[:, n.generators_t['p_max_pu'].columns.str.contains("cluster")]

Generator,DK0 0 0 solar-hsat cluster,DK0 0 4 solar cluster,DK0 0 3 solar cluster,DK0 0 4 onwind cluster,DK1 0 0 solar-hsat cluster,DK1 0 4 solar cluster,DK1 0 3 solar cluster,DK1 0 4 onwind cluster
snapshot,,,,,,,,
2013-01-01 00:00:00,0.050814,0.063981,0.045222,0.696333,0.014524,0.008680,0.009358,0.979361
2013-01-01 12:00:00,0.025825,0.023568,0.020270,0.826427,0.015597,0.004641,0.003397,0.762600
2013-01-02 00:00:00,0.068377,0.081917,0.074602,0.804040,0.059279,0.024506,0.027854,0.935317
2013-01-02 12:00:00,0.027083,0.029115,0.027876,0.942114,0.039273,0.007522,0.029269,0.979369
2013-01-03 00:00:00,0.034479,0.053746,0.041999,0.979890,0.027476,0.013083,0.020153,0.999986
...,...,...,...,...,...,...,...,...
2013-12-29 12:00:00,0.015200,0.023769,0.021790,0.837427,0.012586,0.006311,0.004552,0.992750
2013-12-30 00:00:00,0.031613,0.045437,0.032971,0.943882,0.034032,0.036115,0.028677,0.994667
2013-12-30 12:00:00,0.019495,0.007804,0.015342,0.999543,0.021868,0.013289,0.020721,0.978904


**Exporting**

In [202]:
n.export_to_netcdf(fn)


INFO:pypsa.network.io:Exported network 'Unnamed Network'saved to 'resources/DK_test/networks/base_s_2__12h_2050.nc contains: loads, stores, links, generators, global_constraints, carriers, buses


<xarray.Dataset> Size: 541kB
Dimensions:                               (snapshots: 730,
                                           investment_periods: 0, loads_i: 37,
                                           loads_t_p_set_i: 10, stores_i: 32,
                                           stores_t_e_min_pu_i: 2,
                                           stores_t_e_max_pu_i: 4,
                                           links_i: 145,
                                           links_t_efficiency_i: 8,
                                           links_t_p_max_pu_i: 4,
                                           generators_i: 65,
                                           generators_t_p_max_pu_i: 50,
                                           global_constraints_i: 3,
                                           carriers_i: 118, buses_i: 66)
Coordinates: (12/15)
  * snapshots                             (snapshots) int64 6kB 0 1 ... 728 729
  * investment_periods                    (investment_periods) object 0B 
  * loads_i                               (loads_i) object 296B 'DK0 0' ... '...
  * loads_t_p_set_i                       (loads_t_p_set_i) object 80B 'DK0 0...
  * stores_i                              (stores_i) object 256B 'co2 atmosph...
  * stores_t_e_min_pu_i                   (stores_t_e_min_pu_i) object 16B 'D...
    ...                                    ...
  * links_t_p_max_pu_i                    (links_t_p_max_pu_i) object 32B 'DK...
  * generators_i                          (generators_i) object 520B 'DK0 0 0...
  * generators_t_p_max_pu_i               (generators_t_p_max_pu_i) object 400B ...
  * global_constraints_i                  (global_constraints_i) object 24B '...
  * carriers_i                            (carriers_i) object 944B 'AC' ... '...
  * buses_i                               (buses_i) object 528B 'DK0 0' ... '...
Data variables: (12/90)
    snapshots_snapshot                    (snapshots) datetime64[ns] 6kB 2013...
    snapshots_objective                   (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_stores                      (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_generators                  (snapshots) float64 6kB 12.0 ... 12.0
    investment_periods_objective          (investment_periods) float64 0B 
    investment_periods_years              (investment_periods) float64 0B 
    ...                                    ...
    buses_unit                            (buses_i) object 528B 'MWh_el' ... ...
    buses_location                        (buses_i) object 528B 'DK0 0' ... '...
    buses_control                         (buses_i) object 528B 'Slack' ... 'PQ'
    buses_country                         (buses_i) object 528B 'DK' ... 'DK'
    buses_substation_lv                   (buses_i) float64 528B 1.0 1.0 ... nan
    buses_substation_off                  (buses_i) float64 528B 1.0 1.0 ... nan
Attributes:
    network__multi_invest:  0
    network_name:           Unnamed Network
    network_pypsa_version:  0.35.2
    network_srid:           4326
    crs:                    {"_crs": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"World Geo...
    meta:                   {"version": "v2025.07.0", "tutorial": false, "log...